# New RRL Selection

In this notebook, I will select potential RRL's in SV 95 to fit for in new_RRL_results notebook

## Import Libraries and Load Data

In [ ]:
#import libraries and set up tap service
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.lines as mlines
import numpy as np
from astropy.table import Table
import pyvo
import astropy.units as u
from lsst.rsp import get_tap_service
import pandas as pd
from astroquery.simbad import Simbad
from astropy.coordinates import SkyCoord
from matplotlib.path import Path


from matplotlib.patches import Polygon
pd.set_option('display.max_columns', None)

from lsst.utils.plotting import (get_multiband_plot_colors,
                                 get_multiband_plot_symbols)

from lsst.daf.butler import Butler
from lsst.utils.plotting import (
    get_multiband_plot_colors,
    get_multiband_plot_symbols,
    get_multiband_plot_linestyles
)

service = get_tap_service("tap")
assert service is not None

import seaborn as sns
my_pal = sns.color_palette("colorblind")

import warnings
warnings.filterwarnings('error')

import lsst.afw.display as afwDisplay
import lsst.geom as geom
import lsst.sphgeom as sphgeom
import lsst.afw.display as afw_display
from astropy.wcs import WCS
import numpy as np
import gc

In [ ]:
Ex_diaObj_df = pd.read_csv("data/Extra_diaObj_df.csv",index_col=0)
forcedDiaObj_df = pd.read_csv("data/diaForced_df.csv",index_col=0)

grouped_forcedDia = forcedDiaObj_df.groupby("diaObjectId")

filter_names = ['u', 'g', 'r', 'i', 'z', 'y']
filter_colors = get_multiband_plot_colors()
filter_symbols = get_multiband_plot_symbols()
filter_linestyles = get_multiband_plot_linestyles()

In [ ]:
# Creating New Folder for phase curve plots
outdir = os.getcwd()
search_path = os.path.join(outdir,"RRL_search_results")

if not os.path.isdir(search_path):
    os.mkdir(search_path)

In [ ]:
afwDisplay.setDefaultBackend("matplotlib")

butler = Butler("dp1", collections="LSSTComCam/DP1")
assert butler is not None

In [ ]:
#adding in center point of fields from 301_0_DP1
field_centers = {
    "47 Tuc": [6.02, -72.08],
    "SV 38 7": [37.86, 6.98],
    "Fornax": [40.00, -34.45],
    "ECDFS": [53.13, -28.10],
    "EDFS": [59.10, -48.73],
    "SV 95 -25": [95.00, -25.00],
    "Seagull": [106.23, -10.51],
}
field_names = list(field_centers.keys())

In [ ]:
#load in the maglim for each object
u_maglim = butler.get('deepCoadd_psf_maglim_consolidated_map_weighted_mean',
                      band='u')
g_maglim = butler.get('deepCoadd_psf_maglim_consolidated_map_weighted_mean',
                      band='g')
r_maglim = butler.get('deepCoadd_psf_maglim_consolidated_map_weighted_mean',
                      band='r')
i_maglim = butler.get('deepCoadd_psf_maglim_consolidated_map_weighted_mean',
                      band='i')
z_maglim = butler.get('deepCoadd_psf_maglim_consolidated_map_weighted_mean',
                      band='z')
y_maglim = butler.get('deepCoadd_psf_maglim_consolidated_map_weighted_mean',
                      band="y")

## Pulling out GAIA SV 95 -25 RRL's

In [ ]:
overall_mask = Ex_diaObj_df["comb_auto_filter"]
field_mask = Ex_diaObj_df["DR3_field_loc_center"] == "SV 95 -25"

rrl_sv = Ex_diaObj_df[field_mask & overall_mask].copy()
rrl_sv.head()

## Pulling out all Objects in SV 95 -25

In [ ]:
ra_cen,dec_cen = field_centers["SV 95 -25"]
radius = 1

#set up query to search for identified DP1 sources within 1 degree of GAIA RRL's
query = f"""SELECT dp1.coord_ra as ra, dp1.coord_dec as dec, dp1.ObjectId,
        dp1.g_psfMag, dp1.g_psfMagErr,
        dp1.r_psfMag, dp1.r_psfMagErr,
        dp1.u_psfMag, dp1.u_psfMagErr,
        dp1.g_psfFlux, dp1.g_psfFluxErr,
        dp1.r_psfFlux, dp1.r_psfFluxErr,
        dp1.u_psfFlux, dp1.u_psfFluxErr,
        g_extendedness, u_extendedness, r_extendedness,
        g_pixelFlags_inexact_psfCenter, u_pixelFlags_inexact_psfCenter,
        r_pixelFlags_inexact_psfCenter
        FROM dp1.Object as dp1 
        WHERE CONTAINS(POINT('ICRS', dp1.coord_ra, dp1.coord_dec), 
        CIRCLE('ICRS', {ra_cen}, {dec_cen}, {radius}))=1"""

In [ ]:
#call job on query
job = service.submit_job(query) #needs to be astropy table and key is name in query!
#some columns in overall GAIA_table are broken, not sure which!
job.run()
job.wait(phases=['COMPLETED', 'ERROR'])
print('Job phase is', job.phase)
if job.phase == 'ERROR':
    job.raise_if_error()

In [ ]:
assert job.phase == 'COMPLETED'
SV_results = job.fetch_result().to_table()

SV_df = SV_results.to_pandas()
SV_df.head()

In [ ]:
SV_df["g-r"] = SV_df["g_psfMag"] - SV_df["r_psfMag"]
SV_df["u-g"] = SV_df["u_psfMag"] - SV_df["g_psfMag"]

SV_df["g-r_err"] = SV_df["g_psfMagErr"] + SV_df["r_psfMagErr"]
SV_df["u-g_err"] = SV_df["u_psfMagErr"] + SV_df["g_psfMagErr"]

In [ ]:
mask_extend = (SV_df["g_extendedness"] == 0) | (SV_df["u_extendedness"] == 0) | (SV_df["r_extendedness"] == 0)
mask_color_range = (SV_df["u-g"] >= 0.8) & (SV_df["u-g"] <= 1.6) & (SV_df["g-r"] >= -0.4) & (SV_df["g-r"] <= 0.5)

extend_df = SV_df[mask_extend & mask_color_range].copy().reset_index()
extend_df["color_flags"] = "-"

In [ ]:
#set up query to search for identified DP1 sources within 1" of GAIA RRL's
query = """SELECT ext.ObjectId,
        dp1.ra, dp1.dec, dp1.diaObjectId, dp1.nDiaSources,
        dp1.g_psfFluxMean, dp1.u_psfFluxMean, dp1.r_psfFluxMean, 
        dp1.i_psfFluxMean, dp1.z_psfFluxMean, dp1.y_psfFluxMean,
        g_psfFluxSigma, u_psfFluxSigma, r_psfFluxSigma, i_psfFluxSigma, z_psfFluxSigma, y_psfFluxSigma,
        g_scienceFluxMean, u_scienceFluxMean, r_scienceFluxMean, i_scienceFluxMean, z_scienceFluxMean, y_scienceFluxMean,
        g_scienceFluxMeanErr, u_scienceFluxMeanErr, r_scienceFluxMeanErr, i_scienceFluxMeanErr, 
        z_scienceFluxMeanErr, y_scienceFluxMeanErr,
        g_psfFluxChi2, u_psfFluxChi2, r_psfFluxChi2, i_psfFluxChi2, z_psfFluxChi2, y_psfFluxChi2,
        g_psfFluxPercentile25, g_psfFluxPercentile75,
        u_psfFluxPercentile25, u_psfFluxPercentile75,
        r_psfFluxPercentile25, r_psfFluxPercentile75,
        i_psfFluxPercentile25, i_psfFluxPercentile75,
        z_psfFluxPercentile25, z_psfFluxPercentile75,
        y_psfFluxPercentile25, y_psfFluxPercentile75,
        g_psfFluxStetsonJ,u_psfFluxStetsonJ,r_psfFluxStetsonJ,i_psfFluxStetsonJ,z_psfFluxStetsonJ,y_psfFluxStetsonJ
        FROM dp1.DiaObject as dp1 
        JOIN TAP_UPLOAD.extend_table AS ext 
        WHERE CONTAINS(POINT('ICRS', dp1.ra, dp1.dec), 
        CIRCLE('ICRS', ext.ra, ext.dec, 0.00027))=1"""

In [ ]:
#call job on query
extend_table = Table.from_pandas(extend_df[["ObjectId","ra","dec"]], index=False)
job = service.submit_job(query, 
                         uploads={"extend_table": extend_table}) #needs to be astropy table and key is name in query!
#some columns in overall GAIA_table are broken, not sure which!
job.run()
job.wait(phases=['COMPLETED', 'ERROR'])
print('Job phase is', job.phase)
if job.phase == 'ERROR':
    job.raise_if_error()

In [ ]:
assert job.phase == 'COMPLETED'
extend_dia_res = job.fetch_result().to_table()

In [ ]:
extend_dia_df = extend_dia_res.to_pandas()
extend_dia_df['diaObjectId'] = extend_dia_df['diaObjectId'].astype(int)
extend_dia_df['ObjectId'] = extend_dia_df['ObjectId'].astype(int)

unique_extend_dia_df = extend_dia_df.sort_values(by='nDiaSources', ascending=False).drop_duplicates(subset=['ObjectId'], keep='first')
unique_extend_dia_df = unique_extend_dia_df.sort_values(by='nDiaSources', ascending=False).drop_duplicates(subset=['diaObjectId'], keep='first')

In [ ]:
merged_extend_df_temp = pd.merge(unique_extend_dia_df,extend_df,on="ObjectId",how="inner")
merged_extend_df_temp = merged_extend_df_temp.drop(["ra_y","dec_y"],axis=1)
merged_extend_df_temp = merged_extend_df_temp.rename({"ra_x":"ra","dec_x":"dec"},axis=1)

In [ ]:
non_dia_df = extend_df[~extend_df.ObjectId.isin(merged_extend_df_temp.ObjectId)].copy()
non_dia_df["diaObjectId"] = 0
merged_extend_df = pd.concat([merged_extend_df_temp, non_dia_df], axis=0, ignore_index=True)

In [ ]:
merged_extend_df["ObjectId"] = merged_extend_df["ObjectId"].astype(int)
merged_extend_df["diaObjectId"] = merged_extend_df["diaObjectId"].fillna(0)
merged_extend_df["diaObjectId"] = merged_extend_df["diaObjectId"].astype(int)

In [ ]:
loose_merged_extend_df = merged_extend_df.copy()

## Searching for Potential RRL through filters and colors

In [ ]:
#using Jeff's mask with >9 diaSources
sigcut = 0.02
stetsoncut = 10.0
iqrcut = 0.05

in_dia = merged_extend_df["diaObjectId"] != 0


merged_extend_df['r_psfFluxIQR'] = merged_extend_df['r_psfFluxPercentile75'] -\
    merged_extend_df['r_psfFluxPercentile25']
merged_extend_df['r_psfFluxIQRFrac'] = merged_extend_df['r_psfFluxIQR']/extend_dia_df['r_scienceFluxMean']

magrange = (merged_extend_df['r_psfMag'] > 16) & (merged_extend_df['r_psfMag'] < 21.5) #color of 15.5 adds a lot more
pick_color = (merged_extend_df["g-r"] < 0.4) &\
             (merged_extend_df["g-r"] > -0.2)
pick_sig = (merged_extend_df['r_psfFluxSigma']/merged_extend_df['r_scienceFluxMean'] > sigcut)
pick_stetsonj = (merged_extend_df['r_psfFluxStetsonJ'] > stetsoncut)
#no IQR since not always most reliable
dia = merged_extend_df["nDiaSources"] >= 10

in_dia_mask = in_dia

enough_dia_mask = merged_extend_df["diaObjectId"] >= 10

jeff_mask = magrange & pick_color & pick_sig & pick_stetsonj & dia

rrl_mask = merged_extend_df["ObjectId"].isin(rrl_sv["ObjectId"])

In [ ]:
merged_extend_df["shape_flags"] = "-"

merged_extend_df["color_flags"] = "in_obj"
merged_extend_df["shape_flags"] = "in_obj"
merged_extend_df["point_size"] = 5

merged_extend_df.loc[in_dia_mask,"color_flags"] = "in_dia"
merged_extend_df.loc[in_dia_mask,"shape_flags"] = "in_dia"
merged_extend_df.loc[in_dia_mask,"point_size"] = 15

merged_extend_df.loc[jeff_mask,"color_flags"] = "in_passed"
merged_extend_df.loc[jeff_mask,"shape_flags"] = "in_passed"
merged_extend_df.loc[jeff_mask,"point_size"] = 60

merged_extend_df.loc[rrl_mask,"color_flags"] = "in_GAIA"
merged_extend_df.loc[rrl_mask,"shape_flags"] = "in_GAIA"
merged_extend_df.loc[rrl_mask,"point_size"] = 70

In [ ]:
#visualizing
points = np.array([
    [1.3, -0.1],
    [1.12, -0.14],
    [0.97, 0.04],
    [0.97, 0.26],
    [1.18, 0.34],
    [1.3, 0.18]
])

# Create the figure and axes
fig, ax = plt.subplots(figsize=(6,7))

# Create the Polygon patch
# 'closed=True' is default and handles closing the shape automatically
poly = patches.Polygon(points, closed=True, edgecolor='k', facecolor=my_pal[2], alpha=0.5) #

# Add the patch to the axes
ax.add_patch(poly)

p = sns.scatterplot(data=merged_extend_df.sort_values(by='point_size'),x="u-g",y="g-r",size="point_size",
                edgecolor="k", hue="color_flags", hue_order=["in_obj","in_dia","in_passed","in_GAIA"],
                style="color_flags",palette=my_pal[:2] + my_pal[4:5] + my_pal[7:8])

handles, labels = p.get_legend_handles_labels()

plt.legend(handles=handles[:5], labels=labels[:5],loc="lower right")
plt.ylim(-0.4,0.5)
plt.xlim(0.8,1.5)
plt.tight_layout()

plt.savefig(f"{search_path}/color_color.png",dpi=330,bbox_inches="tight")
plt.show()
plt.close()

In [ ]:
x_err = merged_extend_df["g_psfMagErr"] + merged_extend_df["u_psfMagErr"]
y_err = merged_extend_df["r_psfMagErr"] + merged_extend_df["g_psfMagErr"]

error_mask = (x_err < 0.05) & (y_err < 0.05)

In [ ]:
#visualizing w/error mask
points = np.array([
    [1.3, -0.1],
    [1.12, -0.14],
    [0.97, 0.04],
    [0.97, 0.26],
    [1.18, 0.34],
    [1.3, 0.18]
])

# Create the figure and axes
fig, ax = plt.subplots(figsize=(6,7))



# Create the Polygon patch
# 'closed=True' is default and handles closing the shape automatically
poly = patches.Polygon(points, closed=True, edgecolor='k', facecolor=my_pal[2], alpha=0.5) #

# Add the patch to the axes
ax.add_patch(poly)

p = sns.scatterplot(data=merged_extend_df[error_mask].sort_values(by='point_size'),x="u-g",y="g-r",size="point_size",
                edgecolor="k", hue="color_flags", hue_order=["in_obj","in_dia","in_passed","in_GAIA"],
                style="color_flags",palette=my_pal[:2] + my_pal[4:5] + my_pal[7:8])

handles, labels = p.get_legend_handles_labels()

plt.legend(handles=handles[:5], labels=labels[:5],loc="lower right")


plt.ylim(-0.4,0.5)
plt.xlim(0.8,1.5)
plt.tight_layout()

plt.savefig(f"{search_path}/error_color_color.png",dpi=330,bbox_inches="tight")
plt.show()
plt.close()

In [ ]:
reduced_merged_df = merged_extend_df[error_mask & (jeff_mask | in_dia)]
in_polygon_df = reduced_merged_df[(reduced_merged_df["nDiaSources"] >= 10) & 
    (reduced_merged_df["g-r"] < 0.3) & (reduced_merged_df["g-r"] > -0.2) & 
    (reduced_merged_df["u-g"] > 0.9) & (reduced_merged_df["u-g"] < 1.3)]

In [ ]:
#visualizing w/error mask
points = np.array([
    [1.3, -0.1],
    [1.12, -0.14],
    [0.97, 0.04],
    [0.97, 0.26],
    [1.18, 0.34],
    [1.3, 0.18]
])

# Create the figure and axes
fig, ax = plt.subplots(figsize=(6,7))



# Create the Polygon patch
# 'closed=True' is default and handles closing the shape automatically
poly = patches.Polygon(points, closed=True, edgecolor='k', facecolor=my_pal[2], alpha=0.5) #

# Add the patch to the axes
ax.add_patch(poly)

p = sns.scatterplot(data=in_polygon_df.sort_values(by='point_size'),x="u-g",y="g-r",size="point_size",
                edgecolor="k", hue="color_flags", hue_order=["in_obj","in_dia","in_passed","in_GAIA"],
                style="color_flags",style_order=["in_obj","in_dia","in_passed","in_GAIA"],
                    palette=my_pal[:2] + my_pal[4:5] + my_pal[7:8])

handles, labels = p.get_legend_handles_labels()

plt.legend(handles=handles[:5], labels=labels[:5],loc="lower right")


plt.ylim(-0.4,0.5)
plt.xlim(0.8,1.5)
plt.tight_layout()

plt.savefig(f"{search_path}/selected_color_color.png",dpi=330,bbox_inches="tight")
plt.show()
plt.close()

In [ ]:
# simbad = Simbad()
# simbad.add_votable_fields("otype", "allfluxes")

# def look_at_sinbad(row):
#     try:
#         return simbad.query_region(SkyCoord(row['ra'], row['dec'],
#                                       unit=(u.deg, u.deg),
#                                       frame='icrs'), radius=2 * u.arcsec)
#     except:
#         return "N/A"
# sinbad_results = in_polygon_df.apply(look_at_sinbad,axis=1)

In [ ]:
# sinbad_results.value_counts()

In [ ]:
# for i, tab in enumerate(sinbad_results):
#     if tab != "N/A":
#         # print(tab.columns)
#         print(tab["otype"])

## Repeating Selection with more loose mag limits

In [ ]:
#using Jeff's mask with >9 diaSources
sigcut = 0.02
stetsoncut = 10.0
iqrcut = 0.05

in_dia = loose_merged_extend_df["diaObjectId"] != 0


loose_merged_extend_df['r_psfFluxIQR'] = loose_merged_extend_df['r_psfFluxPercentile75'] -\
    loose_merged_extend_df['r_psfFluxPercentile25']
loose_merged_extend_df['r_psfFluxIQRFrac'] = loose_merged_extend_df['r_psfFluxIQR']/extend_dia_df['r_scienceFluxMean']

pick_color = (loose_merged_extend_df["g-r"] < 0.6) &\
             (loose_merged_extend_df["g-r"] > -0.4)
pick_sig = (loose_merged_extend_df['r_psfFluxSigma']/loose_merged_extend_df['r_scienceFluxMean'] > sigcut)
pick_stetsonj = (loose_merged_extend_df['r_psfFluxStetsonJ'] > stetsoncut)
pick_iqr = (loose_merged_extend_df['r_psfFluxIQRFrac'] > iqrcut)
dia = loose_merged_extend_df["nDiaSources"] >= 10

in_dia_mask = in_dia

enough_dia_mask = loose_merged_extend_df["diaObjectId"] >= 10

jeff_mask = pick_color & pick_sig & pick_stetsonj & dia

rrl_mask = loose_merged_extend_df["ObjectId"].isin(rrl_sv["ObjectId"])

In [ ]:
loose_merged_extend_df["shape_flags"] = "-"

loose_merged_extend_df["color_flags"] = "in_obj"
loose_merged_extend_df["shape_flags"] = "in_obj"
loose_merged_extend_df["point_size"] = 5

loose_merged_extend_df.loc[in_dia_mask,"color_flags"] = "in_dia"
loose_merged_extend_df.loc[in_dia_mask,"shape_flags"] = "in_dia"
loose_merged_extend_df.loc[in_dia_mask,"point_size"] = 15

loose_merged_extend_df.loc[jeff_mask,"color_flags"] = "in_passed"
loose_merged_extend_df.loc[jeff_mask,"shape_flags"] = "in_passed"
loose_merged_extend_df.loc[jeff_mask,"point_size"] = 60

loose_merged_extend_df.loc[rrl_mask,"color_flags"] = "in_GAIA"
loose_merged_extend_df.loc[rrl_mask,"shape_flags"] = "in_GAIA"
loose_merged_extend_df.loc[rrl_mask,"point_size"] = 70

In [ ]:
#visualizing
points = np.array([
    [1.3, -0.1],
    [1.12, -0.14],
    [0.97, 0.04],
    [0.97, 0.26],
    [1.18, 0.34],
    [1.3, 0.18]
])

# Create the figure and axes
fig, ax = plt.subplots(figsize=(6,7))

# Create the Polygon patch
# 'closed=True' is default and handles closing the shape automatically
poly = patches.Polygon(points, closed=True, edgecolor='k', facecolor=my_pal[2], alpha=0.5) #

# Add the patch to the axes
ax.add_patch(poly)

p = sns.scatterplot(data=loose_merged_extend_df.sort_values(by='point_size'),x="u-g",y="g-r",size="point_size",
                edgecolor="k", hue="color_flags", hue_order=["in_obj","in_dia","in_passed","in_GAIA"],
                style="color_flags",palette=my_pal[:2] + my_pal[4:5] + my_pal[7:8])

handles, labels = p.get_legend_handles_labels()

plt.legend(handles=handles[:5], labels=labels[:5],loc="lower right")


plt.ylim(-0.4,0.5)
plt.xlim(0.8,1.5)
plt.tight_layout()

plt.savefig(f"{search_path}/looser_color_color.png",dpi=330,bbox_inches="tight")
plt.show()
plt.close()

In [ ]:
x_err = loose_merged_extend_df["g_psfMagErr"] + loose_merged_extend_df["u_psfMagErr"]
y_err = loose_merged_extend_df["r_psfMagErr"] + loose_merged_extend_df["g_psfMagErr"]

error_mask = (x_err < 0.05) & (y_err < 0.05)

In [ ]:
#visualizing w/error mask
points = np.array([
    [1.3, -0.1],
    [1.12, -0.14],
    [0.97, 0.04],
    [0.97, 0.26],
    [1.18, 0.34],
    [1.3, 0.18]
])

# Create the figure and axes
fig, ax = plt.subplots(figsize=(6,7))



# Create the Polygon patch
# 'closed=True' is default and handles closing the shape automatically
poly = patches.Polygon(points, closed=True, edgecolor='k', facecolor=my_pal[2], alpha=0.5) #

# Add the patch to the axes
ax.add_patch(poly)

p = sns.scatterplot(data=loose_merged_extend_df[error_mask].sort_values(by='point_size'),x="u-g",y="g-r",size="point_size",
                edgecolor="k", hue="color_flags", hue_order=["in_obj","in_dia","in_passed","in_GAIA"],
                style="color_flags",palette=my_pal[:2] + my_pal[4:5] + my_pal[7:8])

handles, labels = p.get_legend_handles_labels()

plt.legend(handles=handles[:5], labels=labels[:5],loc="lower right")


plt.ylim(-0.4,0.5)
plt.xlim(0.8,1.5)
plt.tight_layout()

plt.savefig(f"{search_path}/looser_error_color_color.png",dpi=330,bbox_inches="tight")
plt.show()
plt.close()

In [ ]:
loose_reduced_merged_df = loose_merged_extend_df[error_mask & (jeff_mask | in_dia)]
# loose_in_polygon_df = loose_reduced_merged_df[(loose_reduced_merged_df["nDiaSources"] >= 10) & 
#     (loose_reduced_merged_df["g-r"] < 0.3) & (loose_reduced_merged_df["g-r"] > -0.2) & 
#     (loose_reduced_merged_df["u-g"] > 0.9) & (loose_reduced_merged_df["u-g"] < 1.3)]
#visualizing w/error mask
closer_points = np.array([
    [1.3, -0.1],
    [1.12, -0.14],
    [0.95, 0.04],
    [0.97, 0.26],
    [1.18, 0.34],
    [1.3, 0.18]
])


poly_path = Path(closer_points)

mask = poly_path.contains_points(loose_reduced_merged_df[["u-g", "g-r"]].values)


loose_in_polygon_df = loose_reduced_merged_df[(loose_reduced_merged_df["nDiaSources"] >= 10) & mask].copy()
#| (loose_reduced_merged_df["color_flags"] == "in_passed"))]

In [ ]:
#visualizing filtered by polygon
points = np.array([
    [1.3, -0.1],
    [1.12, -0.14],
    [0.97, 0.04],
    [0.97, 0.26],
    [1.18, 0.34],
    [1.3, 0.18]
])

# Create the figure and axes
fig, ax = plt.subplots(figsize=(6,7))



# Create the Polygon patch
# 'closed=True' is default and handles closing the shape automatically
poly = patches.Polygon(points, closed=True, edgecolor='k', facecolor=my_pal[2], alpha=0.5) #

# Add the patch to the axes
ax.add_patch(poly)

p = sns.scatterplot(data=loose_in_polygon_df.sort_values(by='point_size'),x="u-g",y="g-r",size="point_size",
                edgecolor="k", hue="color_flags", hue_order=["in_obj","in_dia","in_passed","in_GAIA"],
                style="color_flags",style_order=["in_obj","in_dia","in_passed","in_GAIA"],
                    palette=my_pal[:2] + my_pal[4:5] + my_pal[7:8])

handles, labels = p.get_legend_handles_labels()

plt.legend(handles=handles[:5], labels=labels[:5],loc="lower right")


plt.ylim(-0.4,0.5)
plt.xlim(0.8,1.5)
plt.tight_layout()

plt.savefig(f"{search_path}/looser_selected_color_color.png",dpi=330,bbox_inches="tight")
plt.show()
plt.close()

## Filtering based on Simbad and Saving Data

In [ ]:
simbad = Simbad()
simbad.add_votable_fields("otype", "allfluxes")

def look_at_sinbad(row):
    try:
        sinbad_results = simbad.query_region(SkyCoord(row['ra'], row['dec'],
                                      unit=(u.deg, u.deg),
                                      frame='icrs'), radius=2 * u.arcsec)

        return sinbad_results["otype"].value
    except:
        return "N/A"

loose_in_polygon_df["simbad_results"] = loose_in_polygon_df.apply(look_at_sinbad,axis=1).copy()

In [ ]:
final_search_df = loose_in_polygon_df[(loose_in_polygon_df["simbad_results"] == "*") |
    (loose_in_polygon_df["simbad_results"] == "N/A")].copy()

In [ ]:
# final_search_df = final_search_df[final_search_df["color_flags"] == "in_passed"]

final_search_df["diaObjectId"] = final_search_df["diaObjectId"].astype(int)

In [ ]:
# final_search_df["diaObjectId"]

In [ ]:
# final_search_df.shape

In [ ]:
# final_search_df.to_csv("data/rrl_search_results.csv")

## Matching to ForcedSource, Adding Extra Columns, and Saving Data

### Loading Needed Functions

In [ ]:
#define a function to pull out forced sources for each found object
def search_for_flux(row):

    if "ObjectId" in row.index:
        ObjID = row["ObjectId"]
        query = """SELECT fsodo.coord_ra, fsodo.coord_dec,
            fsodo.ObjectId, fsodo.visit, fsodo.band,
            fsodo.psfDiffFlux, fsodo.psfDiffFluxErr,
            fsodo.psfFlux as psfFlux, fsodo.psfFluxErr,
            vis.expMidptMJD,
            pixelFlags_bad, pixelFlags_saturated, pixelFlags_saturatedCenter,
            pixelFlags_suspect, pixelFlags_suspectCenter
            FROM dp1.ForcedSource as fsodo 
            JOIN dp1.Visit as vis ON vis.visit = fsodo.visit
            WHERE fsodo.objectId = """+str(ObjID)
    else:
        print("uh oh")
        

    job = service.submit_job(query)
    job.run()
    job.wait(phases=['COMPLETED', 'ERROR'])
    # print('Job phase is', job.phase)
    if job.phase == 'ERROR':
        job.raise_if_error()

    assert job.phase == 'COMPLETED'
    forced_source = job.fetch_result().to_table()

    forced_df = forced_source.to_pandas()

    forced_df["query"] = query
    forced_df["nSources"] = len(forced_source)

    back_sat = ~forced_df["pixelFlags_saturated"]
    cent_sat = ~forced_df["pixelFlags_saturatedCenter"]
    bad_pixel = ~forced_df["pixelFlags_bad"]

    mask = back_sat & cent_sat & bad_pixel

    forced_df["nGoodSources"] = len(forced_df[mask])
    
    for band in filter_names:
        forced_df[f"n{band}"] = np.sum((forced_df['band'] == band))
        forced_df[f"nGood{band}"] = np.sum((forced_df[mask]['band'] == band))

    return forced_df

In [ ]:
def search_dia_flux(row):
    if "diaObjectId" in row.index:
        DiaObjID = row["diaObjectId"]
        query = """SELECT fsodo.coord_ra, fsodo.coord_dec,
            fsodo.diaObjectId, fsodo.visit, fsodo.band, 
            fsodo.psfDiffFlux, fsodo.psfDiffFluxErr, 
            fsodo.psfFlux as psfFlux, fsodo.psfFluxErr, 
            vis.expMidptMJD, 
            pixelFlags_bad, pixelFlags_saturated, pixelFlags_saturatedCenter,
            pixelFlags_suspect, pixelFlags_suspectCenter
            FROM dp1.ForcedSourceOnDiaObject as fsodo
            JOIN dp1.Visit as vis ON vis.visit = fsodo.visit
            WHERE fsodo.diaObjectId = """+str(DiaObjID)

    else:
        print("uh oh")

    job = service.submit_job(query)
    job.run()
    job.wait(phases=['COMPLETED', 'ERROR'])
    # print('Job phase is', job.phase)
    if job.phase == 'ERROR':
        job.raise_if_error()

    assert job.phase == 'COMPLETED'
    forced_source = job.fetch_result().to_table()

    forced_df = forced_source.to_pandas()

    forced_df["query"] = query
    forced_df["nSources"] = len(forced_source)

    back_sat = ~forced_df["pixelFlags_saturated"]
    cent_sat = ~forced_df["pixelFlags_saturatedCenter"]
    bad_pixel = ~forced_df["pixelFlags_bad"]

    mask = back_sat & cent_sat & bad_pixel

    forced_df["nGoodSources"] = len(forced_df[mask])
    
    for band in filter_names:
        forced_df[f"n{band}"] = np.sum((forced_df['band'] == band))
        forced_df[f"nGood{band}"] = np.sum((forced_df[mask]['band'] == band))

    return forced_df

In [ ]:
def move_columns(df):
    if "ObjectId" in df.columns:
        removed_info = df[["ObjectId","nGoodSources","nSources","query","nu",
                           "ng","nr","ni","nz","ny","nGoodu","nGoodg","nGoodr",
                           "nGoodi","nGoodz","nGoody"]].drop_duplicates()
        df.drop(columns=["nSources","query","nu","ng","nr","ni","nz","ny",
                         "nGoodu","nGoodg","nGoodr","nGoodi","nGoodz","nGoody"], inplace=True)
        return removed_info
    else:
        print("uh oh")
        return None

In [ ]:
def move_dia_columns(df):
    if "diaObjectId" in df.columns:
        print(df["diaObjectId"])
        removed_info = df[["diaObjectId","nSources","nGoodSources","query","nu",
                           "ng","nr","ni","nz","ny","nGoodu","nGoodg","nGoodr",
                           "nGoodi","nGoodz","nGoody"]].drop_duplicates()
        df.drop(columns=["nSources","nGoodSources","query","nu","ng","nr","ni","nz","ny",
                        "nGoodu","nGoodg","nGoodr","nGoodi","nGoodz","nGoody"], inplace=True)
        return removed_info

In [ ]:
#find the r and g average magnitude after converting flux through AB mag
def find_r_group_avg(row):
    diaObjId = row["diaObjectId"]
    
    # group = grouped_forcedSearch.get_group(ObjID)
    group = grouped_diaSearch.get_group(diaObjId)

    flux_vals = group[group["band"] == "r"]["psfFlux"]
    flux_vals = flux_vals[flux_vals > 0]
    mag = -2.5 * np.log10( flux_vals / (3631 * 1e9))

    return np.mean(mag)


def filt_find_r_group_avg(row):
    diaObjId = row["diaObjectId"]
    
    # group = grouped_forcedSearch.get_group(ObjID)
    group = grouped_diaSearch.get_group(diaObjId)
    
    back_sat = ~group["pixelFlags_saturated"]
    cent_sat = ~group["pixelFlags_saturatedCenter"]
    bad_pixel = ~group["pixelFlags_bad"]

    mask = back_sat & cent_sat & bad_pixel

    masked_group = group[mask]

    flux_vals = masked_group[masked_group["band"] == "r"]["psfFlux"]
    flux_vals = flux_vals[flux_vals > 0]
    mag = -2.5 * np.log10( flux_vals / (3631 * 1e9))

    return np.mean(mag)


def find_g_group_avg(row):
    diaObjId = row["diaObjectId"]
    
    # group = grouped_forcedSearch.get_group(ObjID)
    group = grouped_diaSearch.get_group(diaObjId)

    flux_vals = group[group["band"] == "g"]["psfFlux"]
    flux_vals = flux_vals[flux_vals > 0]
    mag = -2.5 * np.log10( flux_vals / (3631 * 1e9))

    return np.mean(mag)


def filt_find_g_group_avg(row):
    diaObjId = row["diaObjectId"]
    
    # group = grouped_forcedSearch.get_group(ObjID)
    group = grouped_diaSearch.get_group(diaObjId)
    
    back_sat = ~group["pixelFlags_saturated"]
    cent_sat = ~group["pixelFlags_saturatedCenter"]
    bad_pixel = ~group["pixelFlags_bad"]

    mask = back_sat & cent_sat & bad_pixel

    masked_group = group[mask]

    flux_vals = masked_group[masked_group["band"] == "g"]["psfFlux"]
    flux_vals = flux_vals[flux_vals > 0]
    mag = -2.5 * np.log10( flux_vals / (3631 * 1e9))

    return np.mean(mag)


def find_u_group_avg(row):
    diaObjId = row["diaObjectId"]
    
    # group = grouped_forcedSearch.get_group(ObjID)
    group = grouped_diaSearch.get_group(diaObjId)

    flux_vals = group[group["band"] == "u"]["psfFlux"]
    flux_vals = flux_vals[flux_vals > 0]
    mag = -2.5 * np.log10( flux_vals / (3631 * 1e9))

    return np.mean(mag)

def filt_find_u_group_avg(row):
    diaObjId = row["diaObjectId"]
    
    # group = grouped_forcedSearch.get_group(ObjID)
    group = grouped_diaSearch.get_group(diaObjId)

    back_sat = ~group["pixelFlags_saturated"]
    cent_sat = ~group["pixelFlags_saturatedCenter"]
    bad_pixel = ~group["pixelFlags_bad"]

    mask = back_sat & cent_sat & bad_pixel

    masked_group = group[mask]

    flux_vals = masked_group[masked_group["band"] == "u"]["psfFlux"]
    flux_vals = flux_vals[flux_vals > 0]
    mag = -2.5 * np.log10( flux_vals / (3631 * 1e9))

    return np.mean(mag)

### Calling Above Functions

In [ ]:
import warnings
# This silences the ResourceWarning about unclosed sockets
warnings.filterwarnings("ignore", category=ResourceWarning)

forcedSources = final_search_df.apply(search_for_flux,axis=1)
forcedDiaSources = final_search_df.apply(search_dia_flux,axis=1)

In [ ]:
forcedSources_dfs_temp = pd.Series([df.copy() for df in list(forcedSources)])
diaSources_dfs_temp = pd.Series([df.copy() for df in list(forcedDiaSources)])


dup_vals = pd.concat(forcedSources_dfs_temp.apply(move_columns).to_list())
search_df = pd.merge(final_search_df,dup_vals,on="ObjectId")

In [ ]:
forcedSources_dfs_long = pd.concat(list(forcedSources_dfs_temp))
grouped_forcedSearch = forcedSources_dfs_long.groupby("ObjectId")

diaSources_dfs_long = pd.concat(list(diaSources_dfs_temp))
grouped_diaSearch = diaSources_dfs_long.groupby("diaObjectId")

In [ ]:
def check_flat_bands(row):
    diaObjId = row["diaObjectId"]
    
    # group = grouped_forcedSearch.get_group(ObjID)
    group = grouped_diaSearch.get_group(diaObjId)

    back_sat = ~group["pixelFlags_saturated"]
    cent_sat = ~group["pixelFlags_saturatedCenter"]
    bad_pixel = ~group["pixelFlags_bad"]

    mask = back_sat & cent_sat & bad_pixel

    masked_group = group[mask]

    max_range = 0

    
    for band in filter_names:
        flux_vals = masked_group[masked_group["band"] == band]["psfFlux"]
        flux_vals = flux_vals[flux_vals > 0]
        mag = -2.5 * np.log10( flux_vals / (3631 * 1e9))

        mag_range = np.max(mag) - np.min(mag)


        if mag_range > max_range:
            max_range = mag_range

    return max_range

def mag_range_flag(col):
    if col > 0.1:
        return True

    return False



search_df["mag_var"] = search_df.apply(check_flat_bands,axis=1)
search_df["mag_var_flag"] = search_df["mag_var"].apply(mag_range_flag)

In [ ]:
# search_df[search_df["mag_var_flag"]]["diaObjectId"]

# 0    614435753623027782
# 2    614436372098318338

In [ ]:
search_df["mag_var_flag"].value_counts()

In [ ]:
refined_search_df = search_df[search_df["mag_var_flag"]].copy()

In [ ]:
refined_search_df.shape

In [ ]:
#call above functions and save new columns to use
mean_mags_r = refined_search_df.apply(find_r_group_avg,axis=1)
mean_mags_g = refined_search_df.apply(find_g_group_avg,axis=1)
mean_mags_u = refined_search_df.apply(find_u_group_avg,axis=1)

refined_search_df["mean_r_mag"] = mean_mags_r
refined_search_df["mean_g_mag"] = mean_mags_g
refined_search_df["mean_u_mag"] = mean_mags_u

In [ ]:
#call above functions (filt) and save new columns to use
mean_mags_r_filt = refined_search_df.apply(filt_find_r_group_avg,axis=1)
mean_mags_g_filt = refined_search_df.apply(filt_find_g_group_avg,axis=1)
mean_mags_u_filt = refined_search_df.apply(filt_find_u_group_avg,axis=1)

refined_search_df["filt_mean_r_mag"] = mean_mags_r_filt
refined_search_df["filt_mean_g_mag"] = mean_mags_g_filt
refined_search_df["filt_mean_u_mag"] = mean_mags_u_filt

In [ ]:
refined_search_df["filt_g-r"] = refined_search_df["filt_mean_g_mag"] - refined_search_df["filt_mean_r_mag"]
refined_search_df["filt_u-g"] = refined_search_df["filt_mean_u_mag"] - refined_search_df["filt_mean_g_mag"]

refined_search_df["in_Obj"] = (refined_search_df["ObjectId"] != 0)

refined_search_df["diaObjectId"] = refined_search_df["diaObjectId"].astype(int)
refined_search_df["ObjectId"] = refined_search_df["ObjectId"].astype(int)

refined_search_df["DR3_field_loc_center"] = "SV 95 -25"

In [ ]:
forcedSources_dfs_long.to_csv("data/rrl_forcedSource_df.csv")
diaSources_dfs_long.to_csv("data/rrl_diaSource_df.csv")

refined_search_df.to_csv("data/rrl_search_df.csv")

In [ ]:
# search_df[search_df.ObjectId.astype(str).str.endswith("1404")]

## Checking Mags of diaObject align with Object

In [ ]:
#plot mag in Rubin vs GAIA to check for outliers
fig, ax = plt.subplots()

p = sns.scatterplot(refined_search_df, x = refined_search_df["r_psfMag"], y = refined_search_df["filt_mean_r_mag"],
            # style = search_df["DR3_field_loc_center"], style_order = hue_ord,
            legend=True,hue=refined_search_df["nGoodr"],palette = "vlag")

sns.lineplot(x = np.linspace(np.min(refined_search_df["filt_mean_r_mag"]), np.max(refined_search_df["filt_mean_r_mag"]),1000),
         y = np.linspace(np.min(refined_search_df["filt_mean_r_mag"]), np.max(refined_search_df["filt_mean_r_mag"]),1000),
         c = "k",alpha=.5,linestyle=":",label="1:1 Line",legend=True)

handles, labels = p.get_legend_handles_labels()

norm = plt.Normalize(refined_search_df["nGoodr"].min(), refined_search_df["nGoodr"].max())
sm = plt.cm.ScalarMappable(cmap="vlag", norm=norm)
sm.set_array([])

plt.xlabel("Average R Mag from Object",fontsize=16)
plt.ylabel("Average R Mag from diaObject",fontsize=16)
plt.gca().invert_xaxis()
plt.gca().invert_yaxis()
plt.title("R Mag in diaObject vs in Object",fontsize=20)
plt.legend(handles=handles[-1:], labels=labels[-1:])

cax = fig.add_axes([0.92, 0.15, 0.03, 0.7])
p.figure.colorbar(sm,cax=cax)
# plt.savefig(f"{stat_path}/mag_mag_comp.png",dpi=330,bbox_inches="tight")
plt.show()
plt.close()